# Train XGBoost Model
Notebook-friendly training and export pipeline for the XGBoost model.


## 1) Imports


In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier


## 2) Paths and Settings


In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Notebook-safe base directory resolution
BASE_DIR = Path.cwd()
if not (BASE_DIR / 'smart_grid_stability_augmented.csv').exists() and (BASE_DIR / 'project').exists():
    BASE_DIR = BASE_DIR / 'project'

DATA_PATH = BASE_DIR / 'smart_grid_stability_augmented.csv'
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR :', BASE_DIR)
print('DATA_PATH:', DATA_PATH)
print('MODELS_DIR:', MODELS_DIR)


BASE_DIR : d:\Desktop\ML Research Papers\project
DATA_PATH: d:\Desktop\ML Research Papers\project\smart_grid_stability_augmented.csv
MODELS_DIR: d:\Desktop\ML Research Papers\project\models


## 3) Load Data


In [3]:
df = pd.read_csv(DATA_PATH)
df['target'] = (df['stabf'].str.lower() == 'unstable').astype(int)

feature_cols = [c for c in df.columns if c not in ['stab', 'stabf', 'target']]
X = df[feature_cols]
y = df['target']

print('Shape:', df.shape)
print('Features:', len(feature_cols))
display(df.head())


Shape: (60000, 15)
Features: 12


,tau1,tau2,tau3,tau4,p1,p2,p3,p4,g1,g2,g3,g4,stab,stabf,target
0,2.959060,3.079885,8.381025,9.780754,3.763085,-0.782604,-1.257395,-1.723086,0.650456,0.859578,0.887445,0.958034,0.055347,unstable,1
1,9.304097,4.902524,3.047541,1.369357,5.067812,-1.940058,-1.872742,-1.255012,0.413441,0.862414,0.562139,0.781760,-0.005957,stable,0
2,8.971707,8.848428,3.046479,1.214518,3.405158,-1.207456,-1.277210,-0.920492,0.163041,0.766689,0.839444,0.109853,0.003471,unstable,1
3,0.716415,7.669600,4.486641,2.340563,3.963791,-1.027473,-1.938944,-0.997374,0.446209,0.976744,0.929381,0.362718,0.028871,unstable,1
4,3.134112,7.608772,4.943759,9.857573,3.525811,-1.125531,-1.845975,-0.554305,0.797110,0.455450,0.656947,0.820923,0.049860,unstable,1


## 4) Train and Evaluate XGBoost


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE,
)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.92,
    colsample_bytree=0.92,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=1,
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy': float(accuracy_score(y_test, y_pred)),
    'precision': float(precision_score(y_test, y_pred)),
    'recall': float(recall_score(y_test, y_pred)),
    'f1_score': float(f1_score(y_test, y_pred)),
    'roc_auc': float(roc_auc_score(y_test, y_prob)),
}

print('XGBoost metrics:')
print(json.dumps(metrics, indent=2))


XGBoost metrics:
{
  "accuracy": 0.9817777777777777,
  "precision": 0.9806963116166839,
  "recall": 0.9909439219784048,
  "f1_score": 0.9857934857934858,
  "roc_auc": 0.998451952474941
}


## 5) Save Artifact


In [5]:
background_df = X_train.sample(n=min(200, len(X_train)), random_state=RANDOM_STATE).copy()

artifact = {
    'model_name': 'XGBoost',
    'model': xgb_model,
    'feature_cols': feature_cols,
    'metrics': metrics,
    'class_names': ['stable', 'unstable'],
    'background_data': background_df.to_dict(orient='records'),
}

out_path = MODELS_DIR / 'xgboost_model.joblib'
joblib.dump(artifact, out_path)

print(f'Saved model: {out_path}')


Saved model: d:\Desktop\ML Research Papers\project\models\xgboost_model.joblib
